# 🌐 Mini-Project 3: API Data Collection Pipeline

**Objective**: Write a robust Python script to extract data from a public REST API, handle pagination/rate limits, clean the JSON response, and save it as a structured CSV.

**Skills Tested**:
- `requests` library
- JSON parsing
- Error handling and loops
- Converting nested JSON to flat Pandas DataFrames

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

## 1. Connecting to the API
We will use a free, public API: The Pokemon API (PokeAPI). We want to extract base stats for various Pokemon.

In [ ]:
def fetch_pokemon_data(pokemon_id):
    """Fetches data for a single pokemon by ID"""
    url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_id}"
    
    try:
        response = requests.get(url, timeout=5)
        
        # Check if request was successful
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 404:
            print(f"Pokemon {pokemon_id} not found.")
            return None
        else:
            print(f"Error {response.status_code} for ID {pokemon_id}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"Connection error: {e}")
        return None

# Test it
pikachu = fetch_pokemon_data(25)
print("Successfully fetched data for:", pikachu['name'].capitalize() if pikachu else "Failed")

## 2. Parsing the JSON
The API returns deeply nested JSON. We need to write a parser to extract just the fields we care about.

In [ ]:
def parse_pokemon(raw_json):
    """Extracts specific fields from the complex JSON response"""
    if not raw_json:
        return None
        
    # Extract basic info
    parsed = {
        'id': raw_json['id'],
        'name': raw_json['name'].capitalize(),
        'height': raw_json['height'],
        'weight': raw_json['weight'],
        'base_experience': raw_json['base_experience']
    }
    
    # Extract types (could be 1 or 2 types)
    types = [t['type']['name'] for t in raw_json['types']]
    parsed['primary_type'] = types[0] if len(types) > 0 else None
    parsed['secondary_type'] = types[1] if len(types) > 1 else None
    
    # Extract base stats (nested list)
    for stat_obj in raw_json['stats']:
        stat_name = stat_obj['stat']['name']
        base_stat = stat_obj['base_stat']
        parsed[f'stat_{stat_name}'] = base_stat
        
    return parsed

# Test the parser
print("Parsed Data:", parse_pokemon(pikachu))

## 3. The Extraction Pipeline
Loop through IDs to build a dataset. Include politeness delays.

In [ ]:
def build_dataset(start_id, end_id):
    """Runs the pipeline to gather a list of pokemon records"""
    records = []
    
    print(f"Starting extraction from ID {start_id} to {end_id}...")
    
    for i in range(start_id, end_id + 1):
        # Fetch
        raw = fetch_pokemon_data(i)
        
        # Parse
        parsed = parse_pokemon(raw)
        if parsed:
            records.append(parsed)
            
        # Be polite to the API server!
        time.sleep(0.1) 
        
        if i % 10 == 0:
            print(f"Processed {i}...")
            
    return records

# Let's fetch the first 30 pokemon
pokemon_list = build_dataset(1, 30)

## 4. Convert to Pandas and Save
Transform the list of dictionaries into a DataFrame and export to CSV.

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(pokemon_list)

print("Final Dataset Shape:", df.shape)
print(df.head())

# Generate a timestamped filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
filename = f"pokemon_data_{timestamp}.csv"

# Save to CSV
df.to_csv(filename, index=False)
print(f"\nSaved successfully to {filename}")